<a href="https://colab.research.google.com/github/paulo-pires/metricas_classificacao_EMNIST/blob/main/metricas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q tensorflow tensorflow_datasets

In [ ]:
# ----------------------------------------------------------------------
# DESAFIO COMPLETO: CNN no MNIST + Métricas de Classificação
# (AJUSTADO PARA TensorFlow 2.19+ / Keras 3 e Erro 404)
# ----------------------------------------------------------------------

# === 1. Importar Bibliotecas ===
import tensorflow as tf
from tensorflow.keras import layers, models
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import ssl
import sys

# Importar a biblioteca necessária para carregar o dataset
try:
    import tensorflow_datasets as tfds
    print("TensorFlow Datasets (tfds) já instalado.")
except ImportError:
    print("Instalando tensorflow-datasets...")
    # O '-q' (quiet) é para suprimir a saída de instalação
    !pip install -q tensorflow-datasets
    import tensorflow_datasets as tfds

# Correção de SSL (pode ser necessário para baixar o dataset)
ssl._create_default_https_context = ssl._create_unverified_context

print(f"Versão do TensorFlow: {tf.__version__}")

# === 2. Carregar e Preparar o Dataset MNIST (Digits) ===
#
# --- INÍCIO DA CORREÇÃO (Nova Abordagem c/ TFDS) ---
# O link manual (storage.googleapis.com) está quebrado (erro 404).
# A forma correta e moderna de carregar o MNIST é usando a
# biblioteca tensorflow_datasets (tfds).
#
print("Carregando dataset MNIST (digits) via TensorFlow Datasets...")

# Carrega os dados de treino e teste.
# 'batch_size=-1' força o tfds a carregar o dataset inteiro em
# memória como um único tensor (em vez de um 'Dataset' iterável).
# 'as_supervised=True' retorna tuplas (imagem, rótulo).
(train_images, train_labels), (test_images, test_labels) = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    batch_size=-1  # Carrega o dataset inteiro em memória
)

# Converter os Tensors resultantes para NumPy arrays,
# como o resto do script espera.
train_images = train_images.numpy()
train_labels = train_labels.numpy()
test_images = test_images.numpy()
test_labels = test_labels.numpy()

print("Dataset carregado com sucesso.")
# --- FIM DA CORREÇÃO ---
#

print(f"Dados de treino: {train_images.shape}, Rótulos: {train_labels.shape}")
print(f"Dados de teste:  {test_images.shape}, Rótulos: {test_labels.shape}")

# Reshape para adicionar o canal de cor (1=preto e branco)
train_images = train_images.reshape((train_images.shape[0], 28, 28, 1))
test_images = test_images.reshape((test_images.shape[0], 28, 28, 1))

# Normalizar os pixels de [0, 255] para [0, 1]
train_images, test_images = train_images / 255.0, test_images / 255.0

# Definir as classes (0-9)
classes = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]

# === 3. Construir o Modelo (CNN) ===
# (O restante do código é IDÊNTICO ao anterior)
model = models.Sequential()
model.add(layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))
model.add(layers.MaxPooling2D((2, 2)))
model.add(layers.Conv2D(64, (3, 3), activation='relu'))

model.add(layers.Flatten())
model.add(layers.Dense(64, activation='relu'))
model.add(layers.Dense(10, activation='softmax')) # 10 classes

# === 4. Compilar o Modelo ===
model.compile(optimizer='adam',
              loss='sparse_categorical_crossentropy',
              metrics=['accuracy'])

# === 5. Treinar o Modelo ===
print("\nIniciando o treinamento do modelo...")
history = model.fit(train_images, train_labels, epochs=5,
                    validation_data=(test_images, test_labels),
                    verbose=1)
print("Treinamento concluído.")

# === 6. Avaliar Acurácia (Forma Padrão) ===
print("\nAvaliando o modelo no conjunto de teste:")
test_loss, test_acc = model.evaluate(test_images, test_labels, verbose=2)
print(f"\nAcurácia (pelo model.evaluate): {test_acc:.2%}")

# === 7. Gerar Matriz de Confusão (NÃO Normalizada) ===
print("Gerando Matriz de Confusão...")
y_true = test_labels
y_pred_probs = model.predict(test_images)
y_pred = np.argmax(y_pred_probs, axis=1)

con_mat = tf.math.confusion_matrix(labels=y_true, predictions=y_pred).numpy()

# === 8. Plotar a Matriz de Confusão NORMALIZADA ===
con_mat_norm = np.around(con_mat.astype('float') / con_mat.sum(axis=1)[:, np.newaxis], decimals=2)

con_mat_df = pd.DataFrame(con_mat_norm,
                          index = classes,
                          columns = classes)

figure = plt.figure(figsize=(8, 8))
sns.heatmap(con_mat_df, annot=True, cmap=plt.cm.Blues)
plt.tight_layout()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.title('Matriz de Confusão Normalizada (MNIST-Digits)')
plt.show()

# ----------------------------------------------------------------------
# === 9. RESOLUÇÃO DO DESAFIO (MÉTRICAS) ===
# ----------------------------------------------------------------------
print("\n" + "="*50)
print("RESOLUÇÃO DO DESAFIO: CÁLCULO DE MÉTRICAS")
print("="*50 + "\n")

# --- Funções do desafio anterior ---
def calcular_sensibilidade(vp, fn):
    try: return vp / (vp + fn)
    except ZeroDivisionError: return 0.0

def calcular_especificidade(vn, fp):
    try: return vn / (fp + vn)
    except ZeroDivisionError: return 0.0

def calcular_precisao(vp, fp):
    try: return vp / (vp + fp)
    except ZeroDivisionError: return 0.0

def calcular_f_score(precisao, sensibilidade):
    try: return 2 * (precisao * sensibilidade) / (precisao + sensibilidade)
    except ZeroDivisionError: return 0.0

# --- Cálculo da Acurácia Geral (pela matriz) ---
acuracia_geral = np.sum(con_mat.diagonal()) / np.sum(con_mat)
print(f"ACURÁCIA GERAL (calculada da matriz): {acuracia_geral:.2%}")
print("(Deve ser igual ao valor 'model.evaluate' acima)\n")
print("--- Métricas por Classe (One-vs-All) ---")

# --- Cálculo das Métricas por Classe ---
for i in classes:
    VP = con_mat[i, i]
    FP = con_mat[:, i].sum() - VP
    FN = con_mat[i, :].sum() - VP
    VN = con_mat.sum() - (VP + FP + FN)

    sensibilidade = calcular_sensibilidade(VP, FN)
    especificidade = calcular_especificidade(VN, FP)
    precisao = calcular_precisao(VP, FP)
    f_score = calcular_f_score(precisao, sensibilidade)

    print(f"\nClasse: {i}")
    print(f"  Sensibilidade (Recall): {sensibilidade:.2%}")
    print(f"  Especificidade:         {especificidade:.2%}")
    print(f"  Precisão:               {precisao:.2%}")
    print(f"  F-score:                {f_score:.2%}")

TensorFlow Datasets (tfds) já instalado.
Versão do TensorFlow: 2.19.0
Carregando dataset MNIST (digits) via TensorFlow Datasets...


Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

...

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/mnist/incomplete.J8N51R_3.0.1/mnist-test.tfrecord*...:   0%|          | 0/…

Dataset mnist downloaded and prepared to /root/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.
Dataset carregado com sucesso.
Dados de treino: (60000, 28, 28, 1), Rótulos: (60000,)
Dados de teste:  (10000, 28, 28, 1), Rótulos: (10000,)


/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)



Iniciando o treinamento do modelo...
Epoch 1/5
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 62s 33ms/step - accuracy: 0.8909 - loss: 0.3498 - val_accuracy: 0.9808 - val_loss: 0.0585
Epoch 2/5
 361/1875 ━━━━━━━━━━━━━━━━━━━━ 41s 28ms/step - accuracy: 0.9842 - loss: 0.0585